# 3. Planner Agent

The **PlannerAgent** receives a high-level user task and decomposes it into a DAG of sub-tasks with explicit dependencies.

It uses an LLM to produce structured JSON describing the task breakdown.

In [ ]:
from unittest.mock import AsyncMock, MagicMock
import json

from mares.agents.planner_agent import PlannerAgent

# Create a mock LLM factory (no real API key needed)
mock_factory = MagicMock()
mock_factory.generate = AsyncMock()
mock_factory.generate.return_value = json.dumps({
    "tasks": [
        {"id": 1, "task": "Research Celery architecture", "depends_on": []},
        {"id": 2, "task": "Identify common failure modes", "depends_on": []},
        {"id": 3, "task": "Analyze logs for patterns", "depends_on": [1, 2]},
        {"id": 4, "task": "Propose and implement fix", "depends_on": [3]},
    ]
})

agent = PlannerAgent(llm_factory=mock_factory)
plan = await agent.run("Analyze Celery task failures in distributed systems")

print(f"Task: {plan.description}")
for st in plan.sub_tasks:
    print(f"  #{st.id}: {st.task} (depends on: {st.depends_on})")

## How the Prompt Works

The planner is guided by a strict system prompt that demands:
- Valid JSON output only
- 1-indexed integer IDs
- Explicit dependency lists
- Small, independent, collectively exhaustive sub-tasks

This structure ensures the DAG executor can process the output reliably.

In [ ]:
# Inspect the system prompt
print(agent.system_prompt[:300] + "...")

## Error Handling

The planner raises `PlanningError` for:
- Empty input
- Invalid JSON from the LLM
- Missing required fields

In [ ]:
# Test error handling
from mares.utils.exceptions import PlanningError

# Invalid JSON
mock_factory.generate.return_value = "not valid json"
try:
    await agent.run("test")
except PlanningError as e:
    print(f"Correctly caught: {e}")

# Reset
mock_factory.generate.return_value = json.dumps({
    "tasks": [{"id": 1, "task": "Valid task", "depends_on": []}]
})